# Prompt Piper — Per-Operator Analysis: Gemini Flash × GSM8K

Shows each of the 28 transformation operators applied **individually** to a small
sample slice so you can inspect what every rule does, how much it compresses, and
how corpus-fitted vocabulary compares to the built-in defaults.

* **Task**: `gsm8k` — grade-school math, generative scoring via the Gemini API
* **Model**: `gemini-2.0-flash` (change to `gemini-1.5-flash` for a cheaper run)
* **Compress field**: `prompt_0shot` (instruction + question only, shots kept intact)
* **Fitting**: `DatasetVocabularyFitter` on the training split

**Sections**
1. Load model and datasets
2. Inspect a GSM8K sample
3. Fit vocabulary and patterns with Gemini Flash
4. Build combined factory
5. Per-operator transformation analysis
   * 5a — Section detection (`PromptSplitter`)
   * 5b — Per-operator DataFrame (default vs fitted)
   * 5c — Token-delta heatmap
   * 5d — Text-change preview
6. Baseline accuracy
7. Run optimisation — fitted factory
8. Run optimisation — default factory
9. Pareto front: fitted vs default
10. Apply best pipeline to a sample
11. Held-out test evaluation
12. Operator frequency analysis

> **Note on `prompt_0shot` compression**: When `compress_field='prompt_0shot'`, the few-shot
> examples (`prompt`) are sent to the model unchanged; only the instruction + current question
> (stored in `prompt_0shot`) is compressed.  The 0-shot prompt has no shared few-shot prefix
> across samples, so `PromptSplitter` places the entire text in the `instruction` section.
> Instruction operators (O1–O22, 22 operators) are applied; shots operators become no-ops.

In [ ]:
import sys
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv()

import warnings
warnings.filterwarnings('ignore')

import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

pd.set_option('display.max_colwidth', 120)

In [ ]:
MODEL_NAME = 'gemini-2.0-flash'   # change to 'gemini-1.5-flash' for a cheaper run
NUM_FS     = 3          # few-shot examples
BS         = 1          # Gemini API: batch_size must be 1

## 1. Load model and datasets

GSM8K has two splits:
- `train` (~7 500 samples) — used exclusively for fitting
- `test` (~1 319 samples) — first 1 000 for search, last ~319 held out

In [ ]:
from prompt_piper.backends.lm_eval import LMEvalBackend

backend = LMEvalBackend.build(
    model_type='gemini',
    model_name=MODEL_NAME,
    task_name='gsm8k',
    num_fewshot=NUM_FS,
    batch_size=BS,
)

print(f'Task output type : {getattr(backend.task, "OUTPUT_TYPE", "unknown")}')
print(f'Tokenizer type   : {type(backend.tokenizer).__name__}')

In [ ]:
# Training split — fitting only; never used in search or evaluation.
train_data = backend.get_dataset(
    split='train',
    num_fewshot=NUM_FS,
    prompt_field='prompt',
    template_field='prompt_0shot',
).as_list()

# Test split — first 1 000 for search, rest held out.
test_data = backend.get_dataset(
    split='test',
    num_fewshot=NUM_FS,
    prompt_field='prompt',
    template_field='prompt_0shot',
).as_list()

search_data  = test_data[:1000]
heldout_data = test_data[1000:]

print(f'Train samples   : {len(train_data)}')
print(f'Search samples  : {len(search_data)}')
print(f'Held-out samples: {len(heldout_data)}')

## 2. Inspect a GSM8K sample

In [ ]:
sample = train_data[0]
print('Keys:', list(sample.keys()))
print()
print('=== Prompt (full, sent to model) ===')
print(sample['prompt'])
print()
print('=== Prompt 0-shot (compressed field) ===')
print(sample['prompt_0shot'])
print()
print(f'=== Target: {sample["target"]!r} ===')
print()
counts = [backend.tokenizer.count(s['prompt_0shot']) for s in train_data[:200]]
print(f'0-shot token counts over first 200 train samples — '
      f'min={min(counts)}, median={int(np.median(counts))}, max={max(counts)}')

## 3. Fit vocabulary and patterns with Gemini Flash

`DatasetVocabularyFitter` mines the training corpus with `compress_field='prompt_0shot'`
so patterns are tuned to the 0-shot question format rather than the full few-shot prompt.

In [ ]:
from prompt_piper.transformations.fitting import DatasetVocabularyFitter

fitter = DatasetVocabularyFitter(
    train_data,
    prompt_field='prompt',
    target_field='target',
    template_field='prompt_0shot',
    compress_field='prompt_0shot',   # compress only the 0-shot field
    sample_size=500,
    seed=42,
    backend=backend.semantic_backend,
)

print('Fitting vocabulary and patterns on 500 training samples...')
report = fitter.report()
print('Done.')

In [ ]:
print('=== New stopwords (beyond defaults) ===')
print(', '.join(sorted(report['stopwords'])) or '  (none)')

print()
print('=== New abbreviations ===')
for pat, repl in report['abbreviations']:
    print(f'  {pat!r:55s}  ->  {repl!r}')
if not report['abbreviations']:
    print('  (none beyond defaults)')

print()
print('=== New section headings ===')
print(', '.join(repr(s) for s in report['sections']) or '  (none)')

if 'short_synonyms' in report:
    print()
    print('=== Short synonyms (LLM-discovered) ===')
    for pat, repl in report['short_synonyms']:
        print(f'  {pat!r:55s}  ->  {repl!r}')
    if not report['short_synonyms']:
        print('  (none beyond defaults)')

if 'chunk_separator' in report:
    print()
    print(f'=== Chunk separator (LLM-discovered) ===')
    print(f'  {report["chunk_separator"]!r}')

## 4. Build combined factory

Fitted instances take priority; remaining semantic operators get Gemini Flash injected.

In [ ]:
fitted_instances = fitter.fit_all()
llm_semantic_backend = backend.semantic_backend

def combined_factory(cls):
    if cls.id in fitted_instances:
        return fitted_instances[cls.id]
    if (cls.category == 'semantic'
            and 'backend' in inspect.signature(cls.__init__).parameters):
        return cls(backend=llm_semantic_backend)
    return cls()

print('Fitted operator IDs:', sorted(fitted_instances.keys()))

## 5. Per-operator transformation analysis

Each operator is applied **individually and in isolation** to `N_INSPECT` test samples.

`compress_field='prompt_0shot'` means all 0-shot prompts are unique per sample —
there is no shared few-shot prefix.  `PromptSplitter` therefore places the entire
`prompt_0shot` text in the **instruction** section.  The 22 instruction operators
(O1–O22, scoped to `"instruction"`) are applied; shots and data operators become
no-ops on this field.

Two vocabulary variants are compared:

| Variant | Vocabulary source |
|---|---|
| **Default** | Built-in word lists and patterns |
| **Fitted** | Corpus-mined from training split via `DatasetVocabularyFitter` |

In [ ]:
from prompt_piper.backends.lm_eval import LMEvalDataset
from prompt_piper.transformations.sections import PromptSplitter
from prompt_piper.transformations import INSTRUCTION_OPS, SHOT_OPS, DATA_OPS

COMPRESS_FIELD = 'prompt_0shot'
N_INSPECT = 3

# Reload a small slice using the 0-shot field as the compress field.
# No shots_field needed: prompt_0shot has no embedded few-shot examples.
inspect_raw = LMEvalDataset(
    backend.task,
    split='test',
    num_fewshot=NUM_FS,
    prompt_field='prompt',
    template_field='prompt_0shot',
).as_list()[:N_INSPECT]

splitter = PromptSplitter(
    inspect_raw,
    compress_field=COMPRESS_FIELD,
)
sections = splitter.split_all()

print(f'Instruction prefix detected : {bool(splitter._instruction_prefix)}')
print(f'  (0-shot prompts are unique per sample — no shared prefix expected)')
print(f'Output cue detected         : {repr(splitter._output_cue) if splitter._output_cue else "(none)"}')
print()

for i, sec in enumerate(sections):
    print(f'── Sample {i} ─────────────────────────────────────────────')
    for label, text in [
        ('instruction', sec.instruction),
        ('shots',       sec.shots),
        ('input_data',  sec.input_data),
        ('output_cue',  sec.output_cue),
    ]:
        tok = backend.tokenizer.count(text) if text else 0
        preview = repr(text[:80]) + ('...' if len(text) > 80 else '')
        print(f'  {label:<12s}: {tok:4d} tok  {preview}')
    print()

In [ ]:
MAX_PREVIEW = 150

scope_config = [
    ('instruction', INSTRUCTION_OPS, lambda s: s.instruction),
    ('shots',       SHOT_OPS,        lambda s: s.shots),
    ('data',        DATA_OPS,        lambda s: s.input_data),
]

rows = []
for scope_label, op_classes, get_text in scope_config:
    for cls in op_classes:
        default_op = cls()
        fitted_op  = fitted_instances.get(cls.id, cls())
        for si, section in enumerate(sections):
            text = get_text(section).strip()
            if not text:
                continue
            orig_tok    = backend.tokenizer.count(text)
            default_out = default_op(text)
            fitted_out  = fitted_op(text)
            def_tok     = backend.tokenizer.count(default_out)
            fit_tok     = backend.tokenizer.count(fitted_out)
            rows.append({
                'scope':           scope_label,
                'op_id':           cls.id,
                'op_name':         cls.name,
                'sample':          si,
                'orig_tok':        orig_tok,
                'default_tok':     def_tok,
                'fitted_tok':      fit_tok,
                'default_delta':   def_tok - orig_tok,
                'fitted_delta':    fit_tok - orig_tok,
                'default_changed': default_out != text,
                'fitted_changed':  fitted_out  != text,
                'orig_preview':    text[:MAX_PREVIEW],
                'default_preview': default_out[:MAX_PREVIEW],
                'fitted_preview':  fitted_out[:MAX_PREVIEW],
            })

df_ops = pd.DataFrame(rows)
print(f'Per-operator table: {len(df_ops)} rows  '
      f'({df_ops["op_id"].nunique()} operators × {N_INSPECT} samples × scopes)')
print()

summary = (
    df_ops
    .groupby(['scope', 'op_name'])[['default_delta', 'fitted_delta']]
    .mean()
    .round(1)
)
summary.columns = ['Δ tok (default)', 'Δ tok (fitted)']
print('Mean token delta per operator (averaged across inspected samples):')
display(summary)

In [ ]:
n_ops = df_ops[['scope', 'op_name']].drop_duplicates().__len__()
fig, axes = plt.subplots(1, 2, figsize=(16, max(8, n_ops * 0.45 + 2)))

pivot_default = df_ops.pivot_table(
    index=['scope', 'op_name'], columns='sample',
    values='default_delta', aggfunc='mean',
)
pivot_fitted = df_ops.pivot_table(
    index=['scope', 'op_name'], columns='sample',
    values='fitted_delta', aggfunc='mean',
)

all_vals = list(pivot_default.values.flatten()) + list(pivot_fitted.values.flatten())
vmax = max(abs(v) for v in all_vals if not np.isnan(v)) or 1

for ax, pivot, label in [
    (axes[0], pivot_default, 'Default vocabulary'),
    (axes[1], pivot_fitted,  'Fitted vocabulary'),
]:
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r',
                   vmin=-vmax, vmax=vmax)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{s}/{n}' for s, n in pivot.index], fontsize=7)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'Sample {i}' for i in pivot.columns], fontsize=9)
    ax.set_title(label)
    ax.set_xlabel('Sample index')
    plt.colorbar(im, ax=ax, label='Δ tokens  (negative = compression)')

plt.suptitle(
    'Per-operator token delta — Gemini Flash × GSM8K (prompt_0shot)\n'
    '(green = tokens saved, red = tokens added)',
    y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
changed = df_ops[df_ops['default_changed'] | df_ops['fitted_changed']].copy()
changed = changed.sort_values(['scope', 'op_id', 'sample'])

n_changed = changed['op_name'].nunique()
n_total   = df_ops['op_id'].nunique()
print(f'Operators that modified at least one sample: {n_changed} / {n_total}')
print()

pd.set_option('display.max_colwidth', 200)
display(
    changed[[
        'scope', 'op_name', 'sample',
        'orig_tok', 'default_delta', 'fitted_delta',
        'orig_preview', 'default_preview', 'fitted_preview',
    ]].reset_index(drop=True)
)
pd.set_option('display.max_colwidth', 120)

## 6. Baseline accuracy

Measure exact-match accuracy on uncompressed `prompt` — the upper bound.

In [ ]:
baseline_samples = heldout_data[:50]
originals = [s['prompt'] for s in baseline_samples]

scores = backend.evaluator.evaluate_batch(originals, originals, baseline_samples)
baseline_acc = float(np.mean(scores))

print(f'Baseline exact-match accuracy (n=50, uncompressed): {baseline_acc:.3f}')
print('(Typical range for Gemini Flash on GSM8K with 3 shots: 0.85 - 0.95)')

## 7. Run optimisation — fitted factory

Search budget:

| | Value | API calls |
|---|---|---|
| Initial population | pop\_size = 50 | 50 × n\_subset = 5 000 |
| 100 generations | 50 offspring/gen | 500 000 sample evals |

Each sample eval = 1 Gemini API call (batch\_size=1).

In [ ]:
from prompt_piper.optimization import ExperimentConfig

config = ExperimentConfig(
    algorithm='nsga2',
    pop_size=50,
    n_gen=100,
    n_subset=100,
    compress_field='prompt_0shot',
    subset_seed=42,
    seed=42,
)

total_evals = config.pop_size * (1 + config.n_gen)
print(f'Individual evaluations : {total_evals:,}')
print(f'Sample evaluations     : {total_evals * config.n_subset:,}')

In [ ]:
from prompt_piper.optimization.runner import Experiment

print('Running NSGA-II with fitted factory...')
experiment_fitted = Experiment(
    config=config,
    dataset=search_data,
    evaluator=backend.evaluator,
    tokenizer=backend.tokenizer,
    transformation_factory=combined_factory,
)
result_fitted = experiment_fitted.run(
    baselines=['no_compression', 'all_transformations']
)
print(result_fitted.summary())

## 8. Run optimisation — default factory

In [ ]:
print('Running NSGA-II with default factory...')
experiment_default = Experiment(
    config=config,
    dataset=search_data,
    evaluator=backend.evaluator,
    tokenizer=backend.tokenizer,
)
result_default = experiment_default.run(
    baselines=['no_compression', 'all_transformations']
)
print(result_default.summary())

## 9. Pareto front: fitted vs default

In [ ]:
def pareto_df(result, label):
    rows = []
    for bits, names, tokens, perf in zip(
        result.pareto_solutions,
        result.pareto_transformation_names,
        result.avg_tokens,
        result.avg_performance,
    ):
        rows.append({
            'factory':    label,
            'avg_tokens': round(float(tokens), 1),
            'accuracy':   round(float(perf), 3),
            'n_ops':      int(bits.sum()),
            'operators':  ', '.join(names) if names else '(none)',
        })
    return pd.DataFrame(rows).sort_values('avg_tokens')

df_fitted  = pareto_df(result_fitted,  'fitted')
df_default = pareto_df(result_default, 'default')

print('=== Fitted factory Pareto front ===')
display(df_fitted[['avg_tokens', 'accuracy', 'n_ops', 'operators']])
print()
print('=== Default factory Pareto front ===')
display(df_default[['avg_tokens', 'accuracy', 'n_ops', 'operators']])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for res, label, color in [
    (result_fitted,  'Fitted factory (NSGA-II)',  'steelblue'),
    (result_default, 'Default factory (NSGA-II)', 'darkorange'),
]:
    idx = res.avg_tokens.argsort()
    ax.plot(res.avg_tokens[idx], res.avg_performance[idx], 'o-', label=label, zorder=3)

nc = result_fitted.baselines['no_compression']
at = result_fitted.baselines['all_transformations']
ax.scatter(nc.avg_tokens.mean(), nc.avg_performance.mean(),
           marker='*', s=250, color='gray', zorder=5, label='no_compression')
ax.scatter(at.avg_tokens.mean(), at.avg_performance.mean(),
           marker='D', s=80,  color='red',  zorder=5, label='all_transformations')
ax.axhline(baseline_acc, color='gray', linestyle='--', linewidth=0.8,
           label=f'baseline acc ({baseline_acc:.3f})')

ax.set_xlabel('Average tokens (compressed prompt_0shot)')
ax.set_ylabel('Exact-match accuracy (GSM8K)')
ax.set_title('Gemini Flash on GSM8K — Fitted vs Default factory')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

nc_tok = float(nc.avg_tokens.mean())
_, tok_f, perf_f = result_fitted.best_compression()
_, tok_d, perf_d = result_default.best_compression()
print(f'No compression            : {nc_tok:.0f} tokens, acc={baseline_acc:.3f}')
print(f'Best compression (fitted) : {tok_f:.0f} tokens ({tok_f/nc_tok:.1%}), acc={perf_f:.3f}')
print(f'Best compression (default): {tok_d:.0f} tokens ({tok_d/nc_tok:.1%}), acc={perf_d:.3f}')

## 10. Apply best pipeline to a sample

In [ ]:
from prompt_piper.transformations import ALL_TRANSFORMATIONS

bits_comp, tok_comp, perf_comp = result_fitted.best_compression()
print(f'Best-compression solution: {tok_comp:.0f} avg tokens, acc={perf_comp:.3f}')

all_instances = [combined_factory(cls) for cls in ALL_TRANSFORMATIONS]
pipeline = [inst for inst, b in zip(all_instances, bits_comp) if b]

print(f'Operators in best-compression pipeline ({len(pipeline)}):')
for inst in pipeline:
    print(f'  {inst.name}')

sample     = heldout_data[5]
original   = sample['prompt_0shot']
compressed = original
for op in pipeline:
    compressed = op(compressed)

n_orig = backend.tokenizer.count(original)
n_comp = backend.tokenizer.count(compressed)
print(f'\n=== Original 0-shot ({n_orig} tokens) ===')
print(original)
print(f'\n=== Compressed ({n_comp} tokens, {n_comp/n_orig:.1%} of original) ===')
print(compressed)

## 11. Held-out test evaluation

Re-score every Pareto solution on `heldout_data` to check generalisation.

In [ ]:
print(f'Re-scoring {len(result_fitted.pareto_solutions)} Pareto solutions on '
      f'{len(heldout_data)} held-out samples...')

heldout_objectives = experiment_fitted.evaluate_solutions(
    result_fitted.pareto_solutions,
    heldout_data,
)
tokens_heldout = heldout_objectives[:, 0]
perf_heldout   = heldout_objectives[:, 1]
print('Done.')

fig, ax = plt.subplots(figsize=(9, 5))
idx = result_fitted.avg_tokens.argsort()
ax.plot(result_fitted.avg_tokens[idx], result_fitted.avg_performance[idx],
        'o--', color='steelblue', alpha=0.5,
        label=f'Search phase (n_subset={config.n_subset})')
idx2 = tokens_heldout.argsort()
ax.plot(tokens_heldout[idx2], perf_heldout[idx2],
        'o-', color='steelblue', label=f'Held-out test (n={len(heldout_data)})')
ax.scatter(nc.avg_tokens.mean(), nc.avg_performance.mean(),
           marker='*', s=250, color='gray', zorder=5, label='no_compression')
ax.axhline(baseline_acc, color='gray', linestyle='--', linewidth=0.8,
           label=f'baseline acc ({baseline_acc:.3f})')
ax.set_xlabel('Average tokens')
ax.set_ylabel('Exact-match accuracy (GSM8K)')
ax.set_title('Search-phase vs held-out performance — fitted factory')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

gap = float(np.mean(result_fitted.avg_performance) - np.mean(perf_heldout))
print(f'Mean accuracy — search: {np.mean(result_fitted.avg_performance):.3f}  '
      f'held-out: {np.mean(perf_heldout):.3f}  gap: {gap:+.3f}')

## 12. Operator frequency analysis

In [ ]:
counts_fitted  = Counter()
counts_default = Counter()

for names in result_fitted.pareto_transformation_names:
    counts_fitted.update(names)
for names in result_default.pareto_transformation_names:
    counts_default.update(names)

all_ops = sorted(set(counts_fitted) | set(counts_default))

if all_ops:
    fig, axes = plt.subplots(
        1, 2,
        figsize=(14, max(4, len(all_ops) * 0.4 + 1)),
        sharey=True,
    )
    for ax, counts, label, color in [
        (axes[0], counts_fitted,  'Fitted factory',  'steelblue'),
        (axes[1], counts_default, 'Default factory', 'darkorange'),
    ]:
        if counts:
            ops, freqs = zip(*counts.most_common())
            ax.barh(list(ops)[::-1], list(freqs)[::-1], color=color)
        ax.set_xlabel('Frequency on Pareto front')
        ax.set_title(label)
    plt.suptitle('Operator frequency — Gemini Flash / GSM8K', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No operators selected on the Pareto front.')

only_fitted  = set(counts_fitted)  - set(counts_default)
only_default = set(counts_default) - set(counts_fitted)
if only_fitted:
    print('Operators unique to fitted front  :', ', '.join(sorted(only_fitted)))
if only_default:
    print('Operators unique to default front :', ', '.join(sorted(only_default)))